In [1]:
# %% CURE SWEEP ONLY - New Kaggle Notebook

import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
import time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# ============================================================================
# 1. LOAD CHECKPOINT TO GET WORST CONFIG
# ============================================================================

CHECKPOINT_PATH = Path('/kaggle/input/sweep-checkkkpoint/sweep_checkpoint.pkl')  # UPDATE THIS

with open(CHECKPOINT_PATH, 'rb') as f:
    all_results = pickle.load(f)
print(f"✓ Loaded {len(all_results)} results")

worst_result = min(all_results, key=lambda r: r.get('final_test_acc', 0))
worst_config = worst_result['config']

print(f"\nWorst config: n={worst_config['n_per_class']}, d_b={worst_config['d_b']}, "
      f"L={worst_config['L_hidden']}, Test Acc={worst_result['final_test_acc']:.4f}\n")

# ============================================================================
# 2. ARCHITECTURE
# ============================================================================

class BottleneckMLP(nn.Module):
    def __init__(self, d0, K, L_hidden, d_b, arch_type='mlp'):
        super().__init__()
        self.arch_type = arch_type
        self.d_b = d_b
        self.L_hidden = L_hidden
        
        self.projector = nn.Sequential(nn.Linear(3072, d0), nn.ReLU())
        
        if arch_type == 'mlp':
            layers = [nn.Linear(d0, d_b), nn.ReLU()]
            for _ in range(L_hidden - 1):
                layers += [nn.Linear(d_b, d_b), nn.ReLU()]
            layers.append(nn.Linear(d_b, K))
            self.network = nn.Sequential(*layers)
            
        elif arch_type == 'batchnorm':
            layers = [nn.Linear(d0, d_b), nn.BatchNorm1d(d_b), nn.ReLU()]
            for _ in range(L_hidden - 1):
                layers += [nn.Linear(d_b, d_b), nn.BatchNorm1d(d_b), nn.ReLU()]
            layers.append(nn.Linear(d_b, K))
            self.network = nn.Sequential(*layers)
            
        elif arch_type == 'resnet':
            self.projection = nn.Linear(d0, d_b)
            self.residual_blocks = nn.ModuleList([nn.Linear(d_b, d_b) for _ in range(L_hidden)])
            self.output = nn.Linear(d_b, K)
    
    def forward(self, x):
        h = self.projector(x)
        
        if self.arch_type == 'resnet':
            h = self.projection(h)
            for block in self.residual_blocks:
                h = h + torch.relu(block(h))
            return self.output(h)
        else:
            return self.network(h)

# ============================================================================
# 3. DATA
# ============================================================================

def load_data(n_per_class, train=True):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    dataset = torchvision.datasets.CIFAR10(root='./data', train=train, download=True, transform=transform)
    
    if train and n_per_class > 0:
        targets = np.array(dataset.targets)
        indices = []
        for k in range(10):
            class_idx = np.where(targets == k)[0]
            np.random.shuffle(class_idx)
            indices.extend(class_idx[:n_per_class])
        dataset = torch.utils.data.Subset(dataset, indices)
    
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    X, y = next(iter(loader))
    return X.view(len(X), -1).to(DEVICE), y.to(DEVICE)

X_train, y_train = load_data(worst_config['n_per_class'], train=True)
X_test, y_test = load_data(0, train=False)

print(f"Data loaded: Train={len(y_train)}, Test={len(y_test)}")

# ============================================================================
# 4. TRAIN FUNCTION
# ============================================================================

def train_arch(arch_type):
    print(f"\n{'='*60}\nTraining: {arch_type.upper()}\n{'='*60}")
    
    model = BottleneckMLP(
        d0=1000, K=10, L_hidden=worst_config['L_hidden'], 
        d_b=worst_config['d_b'], arch_type=arch_type
    ).to(DEVICE)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=worst_config['wd'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100000)
    criterion = nn.CrossEntropyLoss()
    
    start = time.time()
    for step in range(100000):
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        if step % 1000 == 0:
            acc = (model(X_train).argmax(1) == y_train).float().mean().item()
            print(f"  Step {step:5d} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")
            
            if step > 5000 and loss.item() < 0.01:
                break
    
    model.eval()
    with torch.no_grad():
        train_acc = (model(X_train).argmax(1) == y_train).float().mean().item()
        test_acc = (model(X_test).argmax(1) == y_test).float().mean().item()
    
    elapsed = time.time() - start
    print(f"✓ {arch_type.upper()}: Train={train_acc:.4f}, Test={test_acc:.4f}, Time={elapsed:.1f}s")
    
    return {'arch': arch_type, 'train_acc': train_acc, 'test_acc': test_acc, 
            'gap': train_acc - test_acc, 'time': elapsed}

# ============================================================================
# 5. RUN CURE SWEEP
# ============================================================================

results = []
for arch in ['mlp', 'batchnorm', 'resnet']:
    results.append(train_arch(arch))

# ============================================================================
# 6. RESULTS
# ============================================================================

print("\n" + "="*60)
print("CURE SWEEP RESULTS")
print("="*60)

for r in sorted(results, key=lambda x: x['test_acc'], reverse=True):
    print(f"{r['arch'].upper():12} | Test: {r['test_acc']:.4f} | Gap: {r['gap']:.4f}")

print("="*60)

df = pd.DataFrame(results)
df.to_csv('cure_results.csv', index=False)
print("\n✓ Saved to cure_results.csv")

Device: cuda
✓ Loaded 280 results

Worst config: n=640, d_b=128, L=6, Test Acc=0.0796



100%|██████████| 170M/170M [00:15<00:00, 11.1MB/s]


Data loaded: Train=6400, Test=10000

Training: MLP
  Step     0 | Loss: 2.3041 | Acc: 0.1000
  Step  1000 | Loss: 1.8600 | Acc: 0.2931
  Step  2000 | Loss: 1.7797 | Acc: 0.3241
  Step  3000 | Loss: 2.3026 | Acc: 0.1000
  Step  4000 | Loss: 2.3026 | Acc: 0.1000
  Step  5000 | Loss: 2.3026 | Acc: 0.1000
  Step  6000 | Loss: 2.3026 | Acc: 0.1000
  Step  7000 | Loss: 2.3026 | Acc: 0.1000
  Step  8000 | Loss: 2.3026 | Acc: 0.1000
  Step  9000 | Loss: 2.3026 | Acc: 0.1000
  Step 10000 | Loss: 2.3026 | Acc: 0.1000
  Step 11000 | Loss: 2.3026 | Acc: 0.1000
  Step 12000 | Loss: 2.3026 | Acc: 0.1000
  Step 13000 | Loss: 2.3026 | Acc: 0.1000
  Step 14000 | Loss: 2.3026 | Acc: 0.1000
  Step 15000 | Loss: 2.3026 | Acc: 0.1000
  Step 16000 | Loss: 2.3026 | Acc: 0.1000
  Step 17000 | Loss: 2.3026 | Acc: 0.1000
  Step 18000 | Loss: 2.3026 | Acc: 0.1000
  Step 19000 | Loss: 2.3026 | Acc: 0.1000
  Step 20000 | Loss: 2.3026 | Acc: 0.1000
  Step 21000 | Loss: 2.3026 | Acc: 0.1000
  Step 22000 | Loss: 2.30